In [1]:
print ("hi")

hi


In [3]:
import PyPDF2
import re

def extract_links_from_pdf_pypdf2(pdf_path):
    """
    Extract links from PDF using PyPDF2 by scanning text content
    """
    links = []
    
    try:
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            
            # Regex pattern for URLs
            url_pattern = re.compile(
                r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'
            )
            
            for page_num in range(len(pdf_reader.pages)):
                page = pdf_reader.pages[page_num]
                text = page.extract_text()
                
                # Find all URLs in the text
                found_links = url_pattern.findall(text)
                
                for link in found_links:
                    links.append({
                        'page': page_num + 1,
                        'link': link.strip(),
                        'type': 'text_link'
                    })
                
                # Try to extract annotations/links (if any)
                if '/Annots' in page:
                    annotations = page['/Annots']
                    for annot in annotations:
                        if '/A' in annot and '/URI' in annot['/A']:
                            uri = annot['/A']['/URI']
                            links.append({
                                'page': page_num + 1,
                                'link': uri,
                                'type': 'annotation_link'
                            })
            
            return links
            
    except Exception as e:
        print(f"Error reading PDF: {e}")
        return []

# Example usage
if __name__ == "__main__":
    pdf_file = r"D:\sorabo\modul4\jdid\data\25-124195.pdf"  # Replace with your PDF file path
    links = extract_links_from_pdf_pypdf2(pdf_file)
    
    print(f"Found {len(links)} links:")
    for i, link_info in enumerate(links, 1):
        print(f"{i}. Page {link_info['page']}: {link_info['link']} ({link_info['type']})")

Error reading PDF: argument of type 'IndirectObject' is not iterable
Found 0 links:


In [4]:
import PyPDF2
import re
import sys
from urllib.parse import urlparse, urljoin

def extract_links_from_pdf(pdf_path, ignore_first=True):
    """
    Extract HTTP/HTTPS links from PDF file and optionally ignore the first one
    
    Args:
        pdf_path (str): Path to the PDF file
        ignore_first (bool): Whether to ignore the first link found (default: True)
    
    Returns:
        list: List of extracted links (excluding first one if ignore_first=True)
    """
    all_links = []
    
    try:
        # Open the PDF file
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            
            # Regular expression for HTTP/HTTPS links
            # This pattern captures URLs starting with http:// or https://
            url_pattern = re.compile(
                r'\bhttps?://[^\s<>"\'{}|\\^`\[\]]+',
                re.IGNORECASE
            )
            
            print(f"Processing PDF: {pdf_path}")
            print(f"Total pages: {len(pdf_reader.pages)}")
            print("-" * 50)
            
            # Process each page
            for page_num in range(len(pdf_reader.pages)):
                page = pdf_reader.pages[page_num]
                page_text = page.extract_text()
                
                # Find all links on this page
                page_links = url_pattern.findall(page_text)
                
                for link in page_links:
                    # Clean up the link (remove trailing punctuation)
                    clean_link = link.rstrip('.,;:)!?]}"\'')
                    
                    # Parse to validate it's a proper URL
                    try:
                        parsed = urlparse(clean_link)
                        if parsed.scheme and parsed.netloc:  # Valid URL
                            all_links.append({
                                'url': clean_link,
                                'page': page_num + 1,
                                'domain': parsed.netloc,
                                'scheme': parsed.scheme
                            })
                    except Exception:
                        # Skip invalid URLs
                        continue
            
            print(f"Total links found: {len(all_links)}")
            
            # Filter out the first link if requested
            if ignore_first and len(all_links) > 0:
                print(f"Ignoring first link: {all_links[0]['url']}")
                filtered_links = all_links[1:]
                print(f"Links after ignoring first: {len(filtered_links)}")
                return filtered_links
            else:
                return all_links
                
    except FileNotFoundError:
        print(f"Error: File '{pdf_path}' not found.")
        return []
    except Exception as e:
        print(f"Error reading PDF: {str(e)}")
        return []

def analyze_links(links):
    """
    Analyze and display information about extracted links
    """
    if not links:
        print("No links to analyze.")
        return
    
    print("\n" + "="*60)
    print("LINK ANALYSIS REPORT")
    print("="*60)
    
    # Count by domain
    domains = {}
    for link in links:
        domain = link['domain']
        domains[domain] = domains.get(domain, 0) + 1
    
    print(f"\nTotal unique domains: {len(domains)}")
    
    # Show top domains
    if domains:
        print("\nTop domains found:")
        sorted_domains = sorted(domains.items(), key=lambda x: x[1], reverse=True)[:10]
        for domain, count in sorted_domains:
            print(f"  {domain}: {count} links")
    
    # Group by page
    pages_dict = {}
    for link in links:
        page_num = link['page']
        if page_num not in pages_dict:
            pages_dict[page_num] = []
        pages_dict[page_num].append(link['url'])
    
    print(f"\nLinks by page:")
    for page in sorted(pages_dict.keys()):
        print(f"  Page {page}: {len(pages_dict[page])} links")

def save_links_to_file(links, output_file="extracted_links.txt"):
    """
    Save extracted links to a text file
    """
    try:
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(f"Links extracted from PDF (excluding first link)\n")
            f.write("="*60 + "\n\n")
            
            for i, link in enumerate(links, 1):
                f.write(f"{i}. Page {link['page']}: {link['url']}\n")
                f.write(f"   Domain: {link['domain']}\n")
                f.write("-"*40 + "\n")
        
        print(f"\nLinks saved to: {output_file}")
        return True
    except Exception as e:
        print(f"Error saving to file: {str(e)}")
        return False

def find_specific_links(links, search_terms):
    """
    Find links containing specific search terms
    """
    matching_links = []
    for link in links:
        for term in search_terms:
            if term.lower() in link['url'].lower():
                matching_links.append(link)
                break  # Only add once even if multiple terms match
    
    return matching_links

def main():
    """Main function with user interaction"""
    print("PDF Link Extractor")
    print("-" * 50)
    
    # Get PDF file path
    if len(sys.argv) > 1:
        pdf_file = sys.argv[1]
    else:
        pdf_file = input("Enter the path to your PDF file: ").strip()
    
    if not pdf_file:
        print("No file specified. Exiting.")
        return
    
    # Ask if user wants to ignore first link
    ignore_first = True  # Default is to ignore first link
    response = input("\nDo you want to ignore the first link? (Y/n): ").strip().lower()
    if response == 'n':
        ignore_first = False
    
    # Extract links
    links = extract_links_from_pdf(pdf_file, ignore_first=ignore_first)
    
    if not links:
        print("\nNo links found in the PDF (or only one link was found and ignored).")
        return
    
    # Display results
    print("\n" + "="*60)
    print("EXTRACTED LINKS")
    print("="*60)
    
    for i, link in enumerate(links, 1):
        print(f"{i:3}. Page {link['page']:3}: {link['url']}")
        if i % 10 == 0:  # Pause every 10 links
            input("\nPress Enter to continue...")
    
    # Analyze links
    analyze_links(links)
    
    # Ask about filtering
    filter_choice = input("\nDo you want to search for specific links? (y/N): ").strip().lower()
    if filter_choice == 'y':
        search_terms = input("Enter search terms (comma-separated): ").strip().split(',')
        search_terms = [term.strip() for term in search_terms if term.strip()]
        
        if search_terms:
            matching = find_specific_links(links, search_terms)
            print(f"\nFound {len(matching)} links matching your search:")
            for link in matching:
                print(f"  Page {link['page']}: {link['url']}")
    
    # Save results
    save_choice = input("\nDo you want to save the links to a file? (Y/n): ").strip().lower()
    if save_choice != 'n':
        default_name = pdf_file.replace('.pdf', '_links.txt')
        output_file = input(f"Enter output filename (default: {default_name}): ").strip()
        if not output_file:
            output_file = default_name
        save_links_to_file(links, output_file)

# Alternative: Simple version without all the bells and whistles
def simple_extract(pdf_file, ignore_first=True):
    """Simple function to just extract and print links"""
    with open(pdf_file, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)
        all_links = []
        
        url_pattern = re.compile(r'https?://[^\s<>"\'{}|\\^`\[\]]+', re.IGNORECASE)
        
        for page_num in range(len(pdf_reader.pages)):
            page = pdf_reader.pages[page_num]
            text = page.extract_text()
            
            for link in url_pattern.findall(text):
                clean_link = link.rstrip('.,;:)!?]}"\'')
                all_links.append({
                    'url': clean_link,
                    'page': page_num + 1
                })
        
        # Ignore first link if requested
        if ignore_first and len(all_links) > 0:
            print(f"Ignoring first link: {all_links[0]['url']}\n")
            all_links = all_links[1:]
        
        return all_links

# Example usage function
def example_usage():
    """Example of how to use the extractor"""
    # Method 1: Using the main function
    print("Method 1: Interactive mode")
    # main()  # Uncomment to run
    
    # Method 2: Direct function call
    print("\nMethod 2: Direct function call")
    pdf_path = "your_document.pdf"  # Replace with your PDF path
    
    # Extract links ignoring the first one
    links = extract_links_from_pdf(pdf_path, ignore_first=True)
    
    if links:
        print(f"\nExtracted {len(links)} links (first link ignored):")
        for i, link in enumerate(links, 1):
            print(f"{i}. Page {link['page']}: {link['url']}")
    else:
        print("No links found or only one link was found and ignored.")

# Batch processing function
def process_multiple_pdfs(pdf_files):
    """Process multiple PDF files at once"""
    results = {}
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file}")
        links = extract_links_from_pdf(pdf_file, ignore_first=True)
        results[pdf_file] = {
            'total_links': len(links),
            'links': links
        }
        
        # Save each to separate file
        if links:
            output_file = f"{pdf_file.replace('.pdf', '')}_links.txt"
            save_links_to_file(links, output_file)
    
    return results

if __name__ == "__main__":
    # Run in interactive mode
    main()
    
    # Or use directly:
    pdf_path = "D:\sorabo\modul4\jdid\data\25-124195.pdf"
    # links = extract_links_from_pdf(pdf_path, ignore_first=True)
    # for link in links:
    #     print(f"Page {link['page']}: {link['url']}")

PDF Link Extractor
--------------------------------------------------
Error reading PDF: [Errno 22] Invalid argument: '--f=c:\\Users\\faycal\\AppData\\Roaming\\jupyter\\runtime\\kernel-v3d1ab7b6ac43ffa0a9e4a649ec34cc9204d1d1f40.json'

No links found in the PDF (or only one link was found and ignored).


In [6]:
import PyPDF2
import re

# Open the PDF file
pdf_file = open(r"D:\sorabo\modul4\jdid\data\25-124195.pdf", "rb")  # Replace "your_file.pdf" with your PDF file

# Create PDF reader object
pdf_reader = PyPDF2.PdfReader(pdf_file)

# List to store all links
all_links = []

# Go through each page
for page_num in range(len(pdf_reader.pages)):
    # Get page text
    page = pdf_reader.pages[page_num]
    text = page.extract_text()
    
    # Find all HTTP/HTTPS links
    links = re.findall(r'https?://\S+', text)
    
    # Add each link to our list
    for link in links:
        all_links.append({
            'page': page_num + 1,
            'link': link
        })

# Close the PDF file
pdf_file.close()

# Print all links found
print(f"Total links found: {len(all_links)}")
for i, link_info in enumerate(all_links, 1):
    print(f"{i}. Page {link_info['page']}: {link_info['link']}")

Total links found: 41
1. Page 1: https://www.boamp.fr/pages/avis/?
2. Page 3: https://loire.marches-publics.info//avis/index.cfm?
3. Page 3: https://loire.marches-publics.info//avis/index.cfm?
4. Page 5: https://loire.marches-publics.info//avis/index.cfm?
5. Page 6: https://loire.marches-publics.info//avis/index.cfm?
6. Page 8: https://loire.marches-publics.info//avis/index.cfm?
7. Page 8: https://loire.marches-publics.info//avis/index.cfm?
8. Page 10: https://loire.marches-publics.info//avis/index.cfm?
9. Page 10: https://loire.marches-publics.info//avis/index.cfm?
10. Page 12: https://loire.marches-publics.info//avis/index.cfm?
11. Page 12: https://loire.marches-publics.info//avis/index.cfm?
12. Page 14: https://loire.marches-publics.info//avis/index.cfm?
13. Page 14: https://loire.marches-publics.info//avis/index.cfm?
14. Page 16: https://loire.marches-publics.info//avis/index.cfm?
15. Page 17: https://loire.marches-publics.info//avis/index.cfm?
16. Page 19: https://loire.marches-pu